Your AI, Not Your View: The Bias of LLMs in Investment Analysis
link : https://dl.acm.org/doi/pdf/10.1145/3768292.3770375

제시된 논문(ICAIF '25 발표, ACM DOI: 10.1145/3768292.3770375)은 UNIST, LG AI Research, 플로리다 대학교, LinqAlpha 연구진이 공동 연구한 논문으로, 거대 언어 모델(LLM)이 금융 투자 분석을 수행할 때 드러내는 **내재적 편향(Intrinsic Bias)** 과 **확증 편향(Confirmation Bias)** 을 체계적으로 규명하고 정량화한 연구입니다.

이 논문의 핵심 내용을 연구 배경, 3단계 실험 프레임워크, 세부 결과 및 발견점, 시사점 순으로 상세히 정리해 드립니다.

---

### 1. 연구 배경 및 문제 제기 (Introduction & Background)

* **금융 분야의 지식 충돌 (Knowledge Conflict)** : 금융 환경에서는 모델이 사전 학습 과정에서 습득한 내부 파라메트릭 지식(Parametric Knowledge)과 실시간으로 주어지는 시장 데이터 사이에 불일치가 빈번하게 발생합니다.


* **확증 편향의 발생** : 선행 연구에 따르면 LLM은 상충하는 정보(긍정적 근거와 부정적 근거)가 동시에 주어질 때, 객관적으로 양측을 비교·종합하기보다 자신의 내부 사전 지식에 부합하는 근거에만 집착하는 확증 편향을 보입니다.


* **핵심 위험성** : 금융 기관이나 투자자가 특정 섹터(예: 에너지), 특정 기업 규모(예: 중소형주), 특정 전략(예: 모멘텀)을 타깃으로 리서치를 요청하더라도, 모델 자체의 내재적 선호(예: 기술주 선호)가 사용자의 지시를 압도하여 편향되고 일관성 없는 투자 추천을 내놓을 위험이 존재합니다.


* **핵심 연구 질문 (Research Questions)** :
* **RQ 1** : LLM은 섹터(Sector), 기업 규모(Size), 모멘텀(Momentum) 등 핵심 금융 팩터에 대해 어떤 내재적 편향을 지니는가?


* **RQ 2** : 상충하는 정보가 주어졌을 때 이러한 편향으로 인해 어떤 의사결정 왜곡이 발생하는가?





---

### 2. 3단계 실험 프레임워크 (Methodology)

논문은 환각(Hallucination)에 의한 노이즈를 최소화하기 위해 최근 5년간 S&P 500에 연속 편입된 인지도 높은 427개 종목 집합 $S = \{s_1, s_2, \dots, s_{427}\}$ 을 선정하고, 다음과 같은 3단계 파이프라인을 구축했습니다.

```
[Step 1: 근거 생성] ──> [Step 2: 편향 도출] ──> [Step 3: 편향 검증]
(동일 강도 근거 구축)    (정보 균형 하 편향 측정) (반대 근거 강화 및 번복률 측정)

```

#### Step 1: 근거 생성 (Evidence Generation)

* 평가 대상 모델들과의 정렬 간섭을 배제하기 위해 별도의 독립 모델인 Gemini-2.5-Pro를 사용하여 각 종목별 매수 근거($E_{buy}^{(s)}$)와 매도 근거($E_{sell}^{(s)}$)를 생성했습니다.


* 언어적 구조를 통일하고 가격 변동 강도 매개변수를 $I = 5\%$ 로 고정하여, 모든 매수 근거는 $+I$ , 모든 매도 근거는 $-I$ 의 기대 가격 변동을 명시하도록 설계했습니다:



$$\mathbb{E}[\Delta p^{(s)}] = \begin{cases} +I & (\text{Buy Evidence } e_{buy, i}^{(s)}) \\ -I & (\text{Sell Evidence } e_{sell, i}^{(s)}) \end{cases}$$

* 총 3,416개의 정량적·정성적 근거 풀을 구축했습니다.



#### Step 2: 편향 도출 (Bias Elicitation)

* 동일한 개수($\vert{}E_{buy}^{(s)}\vert{} = \vert{}E_{sell}^{(s)}\vert{}$)와 동일한 강도의 매수/매도 근거를 동시에 제공하는 **균형 프롬프트(Balanced Prompt)** 를 구성하여 정보적 균형 상태(지식 충돌 상황)를 만듭니다.


* 위치 편향(Positional Bias)을 방지하기 위해 근거 제시 순서를 무작위로 섞으며 종목당 $N = 10$ 회 의사결정을 반복하여 매수 결정 횟수 $N_{buy}^{(s)}$ 와 매도 결정 횟수 $N_{sell}^{(s)}$ 를 집계합니다.


* **편향 점수 ($\pi_s$)** 수식:



$$\pi_s = \frac{N_{buy}^{(s)} - N_{sell}^{(s)}}{N_{buy}^{(s)} + N_{sell}^{(s)}} \quad (\pi_s \in [-1, 1])$$


($+1$ 에 가까울수록 강한 매수 편향, $-1$ 에 가까울수록 강한 매도 편향)

#### Step 3: 편향 검증 (Bias Verification)

* 가장 편향 점수가 높게 나타난 그룹($G^*$)의 종목들을 대상으로, 모델의 초기 편향에 반하는 **반대 근거(Counter-evidence)** 를 점진적으로 강화하여 의사결정 번복률(Decision Flip Rate, $\phi$)을 측정합니다.


* **접근법 1 (근거 수량 불균형, $\phi_{s^*}^{vol}$)** : 지지 근거보다 반대 근거의 개수를 더 많이 제시(예: 지지 2개 vs 반대 3개, 지지 1개 vs 반대 3개 등).





$$\phi_{s^*}^{vol} = \frac{N_{flip}^{(s^*)}}{N}$$

* **접근법 2 (근거 강도 불균형, $\phi_{s^*}^{int}$)** : 근거 개수는 1:1로 유지하되, 반대 근거의 가격 변동 강도를 $I + \Delta$ ($\Delta \in \{1\%, 3\%, 5\%, 10\%\}$)로 강화.



$$\phi_{s^*}^{int} = \frac{N_{flip}^{(s^*)}}{N}$$

#### 평가 대상 모델

* 상용 및 오픈소스 프론티어 LLM 6종: GPT-4.1, Gemini-2.5-flash, Qwen3-235B-A22B, Mistral-Small-24B, Llama4-Scout, DeepSeek-V3.

---

### 3. 주요 연구 결과 (Key Findings)

#### (1) 금융 팩터별 내재적 편향 (RQ 1)

* **섹터 편향 (Sector Bias)**:


* 대다수 모델에서 전반적으로 매수를 선호하는 양(+)의 편향 점수가 관찰되었으며, 특히 **기술(Technology) 섹터** 에 대해 가장 높은 매수 선호도를 보였습니다.


* Llama4-Scout(전 섹터 평균 0.8 이상)와 DeepSeek-V3는 거의 모든 섹터에서 극단적인 매수 편향을 나타낸 반면, GPT-4.1은 필수소비재(Consumer Defensive) 등 일부 섹터에서 음(-)의 점수를 기록하며 상대적으로 중립적인 양상을 보였습니다.


* 각 모델의 최고 선호 섹터와 최저 선호 섹터 간의 평균 점수 차이는 모든 모델에서 통계적으로 유의미했습니다 (독립표본 t-검정, $p < 0.001$).




* **기업 규모 편향 (Size Bias)**:


* 427개 기업을 시가총액 기준 4개 분위(Q1: 대형주 ~ Q4: 소형주)로 나누어 분석한 결과, 모든 모델에서 시가총액이 작아질수록 편향 점수가 **단조 감소(Monotonic decline)** 했습니다.


* 즉, 대형주를 체계적으로 과대 선호하고 소형주를 저평가하는 경향이 뚜렷했습니다. 이는 사전 학습 코퍼스 내에서 대형 기업의 언급 빈도와 정보량이 훨씬 풍부한 '인지도 효과(Popularity effect)'에 기인합니다. 최고 분위(Q1)와 최저 분위(Q4)의 차이 역시 전 모델에서 통계적으로 유의했습니다 ($p < 0.001$).




* **모멘텀 vs 역발상 편향 (Momentum vs. Contrarian Bias)**:


* 매수 근거를 모멘텀(추세 추종)으로, 매도 근거를 역발상(평균 회귀)으로 설정하거나 그 반대로 교차 배치하여 승률을 비교했습니다.


* 놀랍게도 평가 대상 **모든 LLM에서 역발상(Contrarian) 전략을 선호하는 현상** 이 유의미하게 확인되었습니다 (카이제곱 검정, $p < 0.01$ 또는 $p < 0.001$). 특히 Qwen3-235B의 경우 역발상 관점의 승률이 70%를 상회했습니다.





#### (2) 상충 정보 하에서의 확증 편향 검증 (RQ 2)

* **수량 증가에 따른 번복률 한계 ($\phi^{vol}$)**:


* 반대 근거만 100% 제시된 상황(지지 0개 vs 반대 2개)에서는 모든 모델이 번복률 1.0에 도달하여 정상적으로 판단을 수정했습니다.


* 그러나 지지 근거가 단 1개라도 포함되는 순간(예: 지지 1개 vs 반대 3개, 지지 2개 vs 반대 3개), 반대 근거가 수적으로 우세함에도 불구하고 의사결정 번복률이 급격히 곤두박질쳤습니다.


* 초기 편향이 강했던 모델(Llama4-Scout, DeepSeek-V3)일수록 반대 근거 다수 환경에서도 자신의 초기 결정을 바꾸지 않는 완고함(Stubbornness)을 보였습니다.




* **강도 강화에 따른 번복률 한계 ($\phi^{int}$)**:


* 반대 근거의 강도를 $\Delta = 10\%$ (즉, $+5\%$ 상승 근거 vs $-15\%$ 하락 근거)까지 극단적으로 높였음에도 불구하고, 대다수 모델의 번복률은 60% 미만에 머물렀습니다.


* 초기 편향이 가장 적었던 Gemini-2.5-flash만이 강도 증가에 따라 80% 수준까지 번복률을 높이며 유연한 적응력을 보였습니다.





#### (3) 의사결정 불확실성과 엔트로피 분석 (Entropy Analysis)

* 모델 내부 토큰 확률 분포로부터 섀넌 엔트로피(Shannon Entropy)를 도출하여 의사결정 시의 인지적 갈등을 수치화했습니다:



$$H(\text{Decision}) = -\sum_{x \in \{buy, sell\}} P(x) \log_2 P(x)$$

* **엔트로피 역전 현상** :
* **균형 프롬프트(2 vs 2)** : 편향이 적은 모델(GPT-4.1)은 외적 근거가 팽팽하여 높은 엔트로피(높은 불확실성)를 보인 반면, 편향이 큰 모델(DeepSeek-V3)은 내부 사전 지식으로 동점을 쉽게 깨버리며 매우 낮은 엔트로피(과도한 확신)를 기록했습니다.


* **불균형 프롬프트(2 vs 3)** : 반대 근거가 우세해지자 패턴이 정반대로 역전되었습니다. GPT-4.1은 다수 근거를 객관적으로 수용하며 엔트로피가 감소한 반면, DeepSeek-V3는 자신의 강한 내부 편향과 외부의 다수 근거 간 충돌로 인해 인지 부조화(Cognitive dissonance)를 겪으며 엔트로피가 급격히 치솟았습니다.





---

### 4. 한계점 및 시사점 (Limitations & Conclusion)

* **연구의 한계점** :
1. 근거 생성을 단일 모델(Gemini-2.5-Pro)에 의존하고 단일 수치 변동률로 단순화하여 현실의 복잡한 비정형 정보를 온전히 반영하지 못했을 가능성이 있습니다.


2. 추론형 모델(Reasoning model)의 경우 근거 불균형 자체를 메타적으로 인식해 우회할 여지가 있습니다.


3. 특정 시점의 스냅샷 데이터셋을 사용한 정적 분석이므로 시간에 따른 편향의 동적 변화를 추적하지 못했습니다.




* **결론 및 시사점** :
* LLM은 결코 중립적인 분석가가 아니며, **'기술주 선호', '대형주 선호', '역발상 전략 선호'** 라는 뚜렷한 내재적 편향을 지니고 있습니다.


* 이러한 편향은 실제 금융 실무에서 모델이 객관적 반대 데이터를 무시하고 확증 편향에 빠지게 만듦으로써 금융 서비스의 신뢰성을 심각하게 저해할 수 있습니다.


* 따라서 자산 배분이나 포트폴리오 구축 등 실제 금융 의사결정에 LLM을 도입할 때는 모델의 내재적 편향을 사전에 감사(Audit)하고 보정하는 체계적인 가드레일이 필수적임을 경고합니다.